# 🛡️ AI Agent for SMS Spam Detection and Intelligent Message Filtering

An end-to-end AI system that classifies SMS messages as **Spam** or **Ham**, compares **DistilBERT**, **BERT**, and **BiLSTM** models, computes confidence and risk scores, detects suspicious patterns, and serves a professional **Streamlit** security dashboard exposed via **ngrok**.

**Run every cell in order, top to bottom.** No manual edits required — the notebook auto-detects your GPU, dataset file, and directory paths.

---


## Phase 1 — Environment Setup
Install dependencies and detect GPU/CPU availability.

In [ ]:
# Install all required libraries (quiet mode)
!pip install -q pandas numpy scikit-learn transformers accelerate sentence-transformers \
    torch tensorflow streamlit pyngrok PyPDF2 nltk matplotlib seaborn wordcloud kaggle

print("✅ All libraries installed successfully.")


In [ ]:
# Core imports used throughout the notebook
import os, sys, json, re, string, random, shutil, zipfile, textwrap, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Python version: {sys.version}")


In [ ]:
# GPU / CPU detection (PyTorch + TensorFlow)
import torch
import tensorflow as tf

torch_gpu_available = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch_gpu_available else "cpu")

tf_gpus = tf.config.list_physical_devices('GPU')
tf_gpu_available = len(tf_gpus) > 0

print("=" * 50)
print("GPU STATUS REPORT")
print("=" * 50)
print(f"GPU Available (PyTorch): {torch_gpu_available}")
if torch_gpu_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU Device: None (using CPU)")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"TensorFlow GPU Available: {tf_gpu_available}")
print("=" * 50)

if torch_gpu_available:
    print(f"✅ Using GPU for training: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected. Falling back to CPU. Training will be slower but will still work.")


## Phase 2 — Project Folder Structure
Create the complete project directory tree up front so every later phase can save into it automatically.

In [ ]:
PROJECT_ROOT = Path("SMS_Spam_AI_Project")

FOLDERS = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "visualizations",
    PROJECT_ROOT / "models" / "distilbert",
    PROJECT_ROOT / "models" / "bert",
    PROJECT_ROOT / "models" / "bilstm",
    PROJECT_ROOT / "results",
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Project folder structure created:")
for folder in FOLDERS:
    print(f"   - {folder}/")


## Phase 3 — Kaggle Dataset Download

Upload your `kaggle.json` API token when prompted (Kaggle → Account → Create New API Token).
The notebook configures the Kaggle API automatically and downloads the SMS Spam Collection dataset.


In [ ]:
from google.colab import files

kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)

if not kaggle_json_path.exists():
    print("Please upload your kaggle.json file (Kaggle -> Account -> Create New API Token).")
    try:
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith(".json"):
                shutil.move(fname, kaggle_json_path)
    except Exception as e:
        print(f"⚠️ File upload failed or was skipped: {e}")

if kaggle_json_path.exists():
    os.chmod(kaggle_json_path, 0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_json_path.parent)
    print("✅ kaggle.json configured successfully.")
else:
    print("❌ kaggle.json not found. Please re-run this cell and upload the file.")


In [ ]:
# Download and extract the SMS Spam Collection dataset
RAW_DIR = PROJECT_ROOT / "data" / "raw"

def download_dataset():
    try:
        os.system(f"kaggle datasets download -d uciml/sms-spam-collection-dataset -p {RAW_DIR}")
        zip_files = list(RAW_DIR.glob("*.zip"))
        if not zip_files:
            print("❌ Dataset download failed — no zip file found. Check your kaggle.json credentials.")
            return False
        for zf in zip_files:
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(RAW_DIR)
        print("✅ Dataset downloaded and extracted successfully.")
        return True
    except Exception as e:
        print(f"❌ Dataset download failed: {e}")
        return False

download_success = download_dataset()

print("\nFiles in raw data directory:")
for f in RAW_DIR.glob("*"):
    print(f"   - {f.name}")


In [ ]:
# Automatically detect the SMS dataset file (handles .csv or .tsv, various encodings)
def detect_dataset_file(raw_dir):
    candidates = list(raw_dir.glob("*.csv")) + list(raw_dir.glob("*.tsv"))
    # Prefer files with 'spam' in the name
    preferred = [f for f in candidates if "spam" in f.name.lower()]
    if preferred:
        return preferred[0]
    if candidates:
        return candidates[0]
    return None

dataset_file = detect_dataset_file(RAW_DIR)

if dataset_file is None:
    print("❌ No dataset file detected. Falling back to a small built-in sample so the notebook can still run.")
    sample_data = {
        "v1": ["ham", "spam", "ham", "spam", "ham"],
        "v2": [
            "Hey, are we still on for lunch tomorrow?",
            "WINNER!! You have been selected to receive a £900 prize reward! Call now!",
            "Don't forget to bring the documents to the meeting.",
            "URGENT! Your mobile number has won £2000. Call 09061701461 to claim.",
            "See you at the gym later.",
        ],
    }
    fallback_path = RAW_DIR / "spam_fallback.csv"
    pd.DataFrame(sample_data).to_csv(fallback_path, index=False)
    dataset_file = fallback_path

print(f"✅ Detected dataset file: {dataset_file.name}")

try:
    df_raw = pd.read_csv(dataset_file, encoding="latin-1")
except Exception:
    df_raw = pd.read_csv(dataset_file, encoding="utf-8", sep="\t", header=None, names=["v1", "v2"])

print(f"Loaded dataset with shape: {df_raw.shape}")
df_raw.head()


## Phase 4 — Dataset Understanding and Exploration

In [ ]:
# Automatically identify the message text column and label column
def identify_columns(df):
    label_col, text_col = None, None
    for col in df.columns:
        sample_vals = df[col].dropna().astype(str).str.lower().unique()[:10]
        if set(sample_vals).issubset({"ham", "spam"}):
            label_col = col
            break
    if label_col is None:
        # fall back to first column
        label_col = df.columns[0]

    remaining = [c for c in df.columns if c != label_col]
    # pick the text column as the one with the largest average string length
    avg_lens = {c: df[c].dropna().astype(str).str.len().mean() for c in remaining if df[c].dtype == object}
    if avg_lens:
        text_col = max(avg_lens, key=avg_lens.get)
    else:
        text_col = remaining[0]
    return label_col, text_col

LABEL_COL, TEXT_COL = identify_columns(df_raw)
print(f"Detected label column: '{LABEL_COL}'")
print(f"Detected message column: '{TEXT_COL}'")

df = df_raw[[LABEL_COL, TEXT_COL]].copy()
df.columns = ["label", "message"]
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:", list(df.columns))
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate records:", df.duplicated().sum())

label_counts = df["label"].str.lower().value_counts()
total = len(df)
ham_count = label_counts.get("ham", 0)
spam_count = label_counts.get("spam", 0)

print("\n" + "=" * 40)
print("DATASET SUMMARY")
print("=" * 40)
print(f"Total Messages: {total}")
print(f"Ham Messages: {ham_count}")
print(f"Spam Messages: {spam_count}")
print(f"Spam Percentage: {spam_count/total*100:.2f}%")
print(f"Ham Percentage: {ham_count/total*100:.2f}%")


In [ ]:
VIZ_DIR = PROJECT_ROOT / "data" / "visualizations"
sns.set_style("whitegrid")

# Spam vs Ham distribution
plt.figure(figsize=(6, 5))
sns.countplot(x=df["label"].str.lower(), palette=["#2E86AB", "#E63946"])
plt.title("Spam vs Ham Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.savefig(VIZ_DIR / "spam_vs_ham_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Message length distribution
df["message_length"] = df["message"].astype(str).apply(len)

plt.figure(figsize=(8, 5))
sns.histplot(df["message_length"], bins=40, color="#2E86AB")
plt.title("Message Length Distribution")
plt.xlabel("Message Length (characters)")
plt.savefig(VIZ_DIR / "message_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Message length by class
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["label"].str.lower(), y=df["message_length"], palette=["#2E86AB", "#E63946"])
plt.title("Message Length by Class")
plt.savefig(VIZ_DIR / "message_length_by_class.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
from collections import Counter

def get_top_words(messages, n=20):
    words = []
    for msg in messages:
        words.extend(re.findall(r"[a-zA-Z']+", str(msg).lower()))
    stopwords = {"the","to","a","i","you","and","is","in","for","of","my","it","on","that","are","me","this",
                 "with","your","have","be","at","not","will","can","so","if","was","but","just","or","as",
                 "do","get","we","he","she","up","out","no","now","how","what","when","then","from","by"}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    return Counter(words).most_common(n)

spam_msgs = df[df["label"].str.lower() == "spam"]["message"]
ham_msgs = df[df["label"].str.lower() == "ham"]["message"]

top_words_overall = get_top_words(df["message"], 20)
top_spam_words = get_top_words(spam_msgs, 20)
top_ham_words = get_top_words(ham_msgs, 20)

print("Most frequent words overall:", top_words_overall[:10])
print("\nTop spam keywords:", top_spam_words[:10])
print("\nTop ham keywords:", top_ham_words[:10])

# Bar chart of most frequent words
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=[c for _, c in top_spam_words[:10]], y=[w for w, _ in top_spam_words[:10]], ax=axes[0], color="#E63946")
axes[0].set_title("Top 10 Spam Keywords")
sns.barplot(x=[c for _, c in top_ham_words[:10]], y=[w for w, _ in top_ham_words[:10]], ax=axes[1], color="#2E86AB")
axes[1].set_title("Top 10 Ham Keywords")
plt.tight_layout()
plt.savefig(VIZ_DIR / "spam_ham_keyword_frequency.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Word clouds for spam and ham
spam_text = " ".join(spam_msgs.astype(str))
ham_text = " ".join(ham_msgs.astype(str))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

spam_wc = WordCloud(width=700, height=500, background_color="white", colormap="Reds").generate(spam_text)
axes[0].imshow(spam_wc, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title("Spam Word Cloud", fontsize=16)

ham_wc = WordCloud(width=700, height=500, background_color="white", colormap="Blues").generate(ham_text)
axes[1].imshow(ham_wc, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title("Ham Word Cloud", fontsize=16)

plt.tight_layout()
plt.savefig(VIZ_DIR / "wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

spam_wc.to_file(str(VIZ_DIR / "spam_wordcloud.png"))
ham_wc.to_file(str(VIZ_DIR / "ham_wordcloud.png"))
print("✅ Word clouds saved.")


## Phase 5 — Data Preprocessing

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords as nltk_stopwords

STOPWORDS = set(nltk_stopwords.words("english"))

def clean_message(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)                      # URLs
    text = re.sub(r"\S+@\S+", " ", text)                                 # emails
    text = re.sub(r"<.*?>", " ", text)                                    # HTML tags
    text = re.sub(r"[^a-z0-9\s]", " ", text)                              # special characters
    text = re.sub(r"\s+", " ", text).strip()                             # normalize whitespace
    return text

records_before = len(df)

df["label"] = df["label"].astype(str).str.lower().str.strip()
df = df[df["label"].isin(["ham", "spam"])]
df["message"] = df["message"].astype(str)
df = df.dropna(subset=["message"])
df = df[df["message"].str.strip() != ""]

missing_handled = records_before - len(df)

df = df.drop_duplicates(subset=["message"])
duplicates_removed = records_before - missing_handled - len(df)

df["clean_message"] = df["message"].apply(clean_message)
df = df[df["clean_message"].str.strip() != ""]

records_after = len(df)

print("=" * 40)
print("PREPROCESSING SUMMARY")
print("=" * 40)
print(f"Records before cleaning: {records_before}")
print(f"Records after cleaning: {records_after}")
print(f"Duplicates removed: {duplicates_removed}")
print(f"Missing/invalid values handled: {missing_handled}")
print(f"Spam messages: {(df['label']=='spam').sum()}")
print(f"Ham messages: {(df['label']=='ham').sum()}")

cleaned_path = PROJECT_ROOT / "sms_cleaned.csv"
df[["label", "message", "clean_message"]].to_csv(cleaned_path, index=False)
print(f"\n✅ Cleaned dataset saved to: {cleaned_path}")
df.head()


## Phase 6 — Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(["ham", "spam"])  # fixed order -> ham=0, spam=1
df["label_encoded"] = label_encoder.transform(df["label"])

label_mapping_df = pd.DataFrame({
    "label": label_encoder.classes_,
    "encoded_value": label_encoder.transform(label_encoder.classes_)
})
label_mapping_path = PROJECT_ROOT / "label_mapping.csv"
label_mapping_df.to_csv(label_mapping_path, index=False)

print("Label mapping:")
print(label_mapping_df)
print(f"\n✅ Label mapping saved to: {label_mapping_path}")


## Phase 7 — Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_message"].values
y = df["label_encoded"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Training set size:   {len(X_train)}  (spam ratio: {y_train.mean():.3f})")
print(f"Validation set size: {len(X_val)}  (spam ratio: {y_val.mean():.3f})")
print(f"Test set size:       {len(X_test)}  (spam ratio: {y_test.mean():.3f})")


## Phase 8 — DistilBERT Spam Classification (`distilbert-base-uncased`)

In [ ]:
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    BertTokenizerFast, BertForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2

class SMSTorchDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

print(f"Config -> max_length: {MAX_LENGTH}, batch_size: {BATCH_SIZE}, epochs: {EPOCHS}, device: {DEVICE}")


In [ ]:
# Tokenize with DistilBERT
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_texts(tokenizer, texts):
    return tokenizer(list(texts), truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_enc_db = tokenize_texts(distilbert_tokenizer, X_train)
val_enc_db = tokenize_texts(distilbert_tokenizer, X_val)
test_enc_db = tokenize_texts(distilbert_tokenizer, X_test)

train_ds_db = SMSTorchDataset(train_enc_db, y_train)
val_ds_db = SMSTorchDataset(val_enc_db, y_val)
test_ds_db = SMSTorchDataset(test_enc_db, y_test)

print("✅ DistilBERT tokenization complete.")


In [ ]:
distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(DEVICE)

distilbert_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=SEED,
    report_to=[],
    fp16=torch_gpu_available,
)

distilbert_trainer = Trainer(
    model=distilbert_model,
    args=distilbert_args,
    train_dataset=train_ds_db,
    eval_dataset=val_ds_db,
    compute_metrics=compute_metrics,
)

distilbert_trainer.train()
print("✅ DistilBERT fine-tuning complete.")


In [ ]:
# Evaluate DistilBERT on the test set
db_test_output = distilbert_trainer.predict(test_ds_db)
db_preds = db_test_output.predictions.argmax(-1)

db_accuracy = accuracy_score(y_test, db_preds)
db_precision, db_recall, db_f1, _ = precision_recall_fscore_support(y_test, db_preds, average="binary", zero_division=0)
db_report = classification_report(y_test, db_preds, target_names=["ham", "spam"], output_dict=True)
db_cm = confusion_matrix(y_test, db_preds)

print("DISTILBERT TEST RESULTS")
print(f"Accuracy:  {db_accuracy:.4f}")
print(f"Precision: {db_precision:.4f}")
print(f"Recall:    {db_recall:.4f}")
print(f"F1-score:  {db_f1:.4f}")
print(f"\nSpam -> Precision: {db_report['spam']['precision']:.4f}, Recall: {db_report['spam']['recall']:.4f}, F1: {db_report['spam']['f1-score']:.4f}")
print(f"Ham  -> Precision: {db_report['ham']['precision']:.4f}, Recall: {db_report['ham']['recall']:.4f}, F1: {db_report['ham']['f1-score']:.4f}")

plt.figure(figsize=(5, 4))
sns.heatmap(db_cm, annot=True, fmt="d", cmap="Blues", xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("DistilBERT Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.savefig(PROJECT_ROOT / "results" / "distilbert_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Save trained DistilBERT model and tokenizer
DISTILBERT_DIR = PROJECT_ROOT / "models" / "distilbert"
distilbert_model.save_pretrained(DISTILBERT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_DIR)
print(f"✅ DistilBERT model and tokenizer saved to: {DISTILBERT_DIR}")


## Phase 9 — BERT Spam Classification (`bert-base-uncased`)

In [ ]:
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

train_enc_bert = tokenize_texts(bert_tokenizer, X_train)
val_enc_bert = tokenize_texts(bert_tokenizer, X_val)
test_enc_bert = tokenize_texts(bert_tokenizer, X_test)

train_ds_bert = SMSTorchDataset(train_enc_bert, y_train)
val_ds_bert = SMSTorchDataset(val_enc_bert, y_val)
test_ds_bert = SMSTorchDataset(test_enc_bert, y_test)

print("✅ BERT tokenization complete.")


In [ ]:
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(DEVICE)

bert_args = TrainingArguments(
    output_dir="./bert_checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    seed=SEED,
    report_to=[],
    fp16=torch_gpu_available,
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=train_ds_bert,
    eval_dataset=val_ds_bert,
    compute_metrics=compute_metrics,
)

bert_trainer.train()
print("✅ BERT fine-tuning complete.")


In [ ]:
bert_test_output = bert_trainer.predict(test_ds_bert)
bert_preds = bert_test_output.predictions.argmax(-1)

bert_accuracy = accuracy_score(y_test, bert_preds)
bert_precision, bert_recall, bert_f1, _ = precision_recall_fscore_support(y_test, bert_preds, average="binary", zero_division=0)
bert_report = classification_report(y_test, bert_preds, target_names=["ham", "spam"], output_dict=True)
bert_cm = confusion_matrix(y_test, bert_preds)

print("BERT TEST RESULTS")
print(f"Accuracy:  {bert_accuracy:.4f}")
print(f"Precision: {bert_precision:.4f}")
print(f"Recall:    {bert_recall:.4f}")
print(f"F1-score:  {bert_f1:.4f}")

plt.figure(figsize=(5, 4))
sns.heatmap(bert_cm, annot=True, fmt="d", cmap="Greens", xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("BERT Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.savefig(PROJECT_ROOT / "results" / "bert_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
BERT_DIR = PROJECT_ROOT / "models" / "bert"
bert_model.save_pretrained(BERT_DIR)
bert_tokenizer.save_pretrained(BERT_DIR)
print(f"✅ BERT model and tokenizer saved to: {BERT_DIR}")


## Phase 10 — BiLSTM Spam Classification (TensorFlow/Keras)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

VOCAB_SIZE = 8000
LSTM_MAX_LEN = 100
EMBED_DIM = 64

keras_tokenizer = KerasTokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
keras_tokenizer.fit_on_texts(X_train)

def to_padded_sequences(texts):
    seqs = keras_tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=LSTM_MAX_LEN, padding="post", truncating="post")

X_train_seq = to_padded_sequences(X_train)
X_val_seq = to_padded_sequences(X_val)
X_test_seq = to_padded_sequences(X_test)

print(f"Vocabulary size (actual): {min(VOCAB_SIZE, len(keras_tokenizer.word_index) + 1)}")
print(f"Training sequence shape: {X_train_seq.shape}")


In [ ]:
bilstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=LSTM_MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

bilstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bilstm_model.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

bilstm_history = bilstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=8,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)
print("✅ BiLSTM training complete.")


In [ ]:
bilstm_probs = bilstm_model.predict(X_test_seq).flatten()
bilstm_preds = (bilstm_probs >= 0.5).astype(int)

bilstm_accuracy = accuracy_score(y_test, bilstm_preds)
bilstm_precision, bilstm_recall, bilstm_f1, _ = precision_recall_fscore_support(y_test, bilstm_preds, average="binary", zero_division=0)
bilstm_report = classification_report(y_test, bilstm_preds, target_names=["ham", "spam"], output_dict=True)
bilstm_cm = confusion_matrix(y_test, bilstm_preds)

print("BiLSTM TEST RESULTS")
print(f"Accuracy:  {bilstm_accuracy:.4f}")
print(f"Precision: {bilstm_precision:.4f}")
print(f"Recall:    {bilstm_recall:.4f}")
print(f"F1-score:  {bilstm_f1:.4f}")

plt.figure(figsize=(5, 4))
sns.heatmap(bilstm_cm, annot=True, fmt="d", cmap="Purples", xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
plt.title("BiLSTM Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.savefig(PROJECT_ROOT / "results" / "bilstm_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import pickle

BILSTM_DIR = PROJECT_ROOT / "models" / "bilstm"
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

bilstm_model.save(BILSTM_DIR / "bilstm_model.h5")
with open(BILSTM_DIR / "tokenizer.pkl", "wb") as f:
    pickle.dump(keras_tokenizer, f)

bilstm_config = {"max_len": LSTM_MAX_LEN, "vocab_size": VOCAB_SIZE}
with open(BILSTM_DIR / "config.json", "w") as f:
    json.dump(bilstm_config, f)

print(f"✅ BiLSTM model and tokenizer saved to: {BILSTM_DIR}")


## Phase 11 — Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["DistilBERT", "BERT", "BiLSTM"],
    "Accuracy": [db_accuracy, bert_accuracy, bilstm_accuracy],
    "Precision": [db_precision, bert_precision, bilstm_precision],
    "Recall": [db_recall, bert_recall, bilstm_recall],
    "F1-Score": [db_f1, bert_f1, bilstm_f1],
})

best_model_row = comparison_df.loc[comparison_df["F1-Score"].idxmax()]
BEST_MODEL_NAME = best_model_row["Model"]
BEST_MODEL_F1 = best_model_row["F1-Score"]

print(comparison_df.to_string(index=False))
print(f"\nBest Model: {BEST_MODEL_NAME}")
print(f"Best F1-Score: {BEST_MODEL_F1*100:.2f}%")

comparison_df.to_csv(PROJECT_ROOT / "results" / "model_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
comparison_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", ax=ax)
plt.title("Model Comparison: DistilBERT vs BERT vs BiLSTM")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "model_comparison_chart.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 12 — Spam Detection Engine, Indicators, Risk Score & Intelligent Filtering

This single cell defines every function that `app.py` reuses: cleaning, per-model prediction, indicator detection, risk scoring, filtering category, and a human-readable summary.

In [ ]:
import numpy as np

SPAM_KEYWORDS = {
    "prize": ["win", "won", "winner", "prize", "lottery", "reward", "jackpot", "claim"],
    "free": ["free", "gift", "bonus", "voucher"],
    "urgent": ["urgent", "immediately", "now", "asap", "act now", "hurry", "limited time", "expire"],
    "financial": ["bank", "account", "credit", "loan", "cash", "money", "payment", "refund", "debit"],
    "otp": ["otp", "verification code", "password", "pin", "verify your account"],
    "promotional": ["offer", "discount", "deal", "sale", "subscribe", "buy now", "click here"],
}

def clean_message_for_inference(text):
    return clean_message(text)

def detect_spam_indicators(message):
    text = str(message)
    lower = text.lower()
    indicators = []

    for category, words in SPAM_KEYWORDS.items():
        if any(w in lower for w in words):
            label = {
                "prize": "Prize/lottery-related keyword detected",
                "free": "Free-offer keyword detected",
                "urgent": "Urgency detected",
                "financial": "Financial/bank keyword detected",
                "otp": "OTP or account-verification request detected",
                "promotional": "Promotional language detected",
            }[category]
            indicators.append(label)

    if re.search(r"https?://|www\.", lower):
        indicators.append("Suspicious URL detected")
    if re.search(r"\b\d{10,}\b", text) or re.search(r"\b0\d{9,}\b", text):
        indicators.append("Phone number detected")
    if re.search(r"[£$€₹]\s?\d", text):
        indicators.append("Currency amount detected")
    letters = [c for c in text if c.isalpha()]
    if letters and sum(1 for c in letters if c.isupper()) / len(letters) > 0.4 and len(letters) > 8:
        indicators.append("Excessive capital letters detected")
    if text.count("!") >= 2 or text.count("?") >= 2:
        indicators.append("Excessive punctuation detected")

    return indicators

def predict_distilbert(message, model, tokenizer, device):
    model.eval()
    inputs = tokenizer(message, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_class = int(np.argmax(probs))
    return {"label": "spam" if pred_class == 1 else "ham", "confidence": float(probs[pred_class])}

def predict_bert(message, model, tokenizer, device):
    return predict_distilbert(message, model, tokenizer, device)  # identical inference path

def predict_bilstm(message, model, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences([message])
    padded = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0).flatten()[0])
    label = "spam" if prob >= 0.5 else "ham"
    confidence = prob if label == "spam" else 1 - prob
    return {"label": label, "confidence": confidence}

def calculate_risk_score(model_results, indicators):
    spam_confidences = [r["confidence"] for r in model_results.values() if r["label"] == "spam"]
    ham_confidences = [r["confidence"] for r in model_results.values() if r["label"] == "ham"]
    n_models = len(model_results)
    spam_votes = len(spam_confidences)

    if spam_votes == 0:
        base_score = (1 - np.mean(ham_confidences)) * 40
    else:
        base_score = (spam_votes / n_models) * 60 + np.mean(spam_confidences) * 30

    indicator_bonus = min(len(indicators) * 4, 15)
    score = min(base_score + indicator_bonus, 100)
    return round(float(score), 2)

def interpret_risk(score):
    if score <= 30:
        return "LOW RISK"
    elif score <= 60:
        return "MEDIUM RISK"
    elif score <= 80:
        return "HIGH RISK"
    else:
        return "VERY HIGH RISK"

def filter_message(final_label, overall_confidence, risk_score,
                    safe_threshold=0.75, spam_threshold=0.75):
    if final_label == "ham" and overall_confidence >= safe_threshold:
        return "SAFE"
    if final_label == "spam" and overall_confidence >= spam_threshold:
        return "SPAM"
    return "SUSPICIOUS"

def generate_recommendation(status, indicators):
    if status == "SPAM":
        return ("Avoid clicking any links in this message. Do not provide OTP, passwords, "
                "bank details, or personal information. Consider blocking the sender.")
    elif status == "SUSPICIOUS":
        return ("This message shows some suspicious characteristics. Verify the sender through "
                "an official channel before responding or clicking any links.")
    else:
        return "This message appears safe based on the trained models."

def analyze_message(message_text, models_bundle):
    """Full pipeline: clean -> predict with all 3 models -> indicators -> risk -> filter -> summary."""
    if not message_text or not str(message_text).strip():
        return {"error": "Empty or invalid message provided."}

    clean_text = clean_message_for_inference(message_text)

    model_results = {
        "DistilBERT": predict_distilbert(clean_text, models_bundle["distilbert_model"],
                                          models_bundle["distilbert_tokenizer"], models_bundle["device"]),
        "BERT": predict_bert(clean_text, models_bundle["bert_model"],
                              models_bundle["bert_tokenizer"], models_bundle["device"]),
        "BiLSTM": predict_bilstm(clean_text, models_bundle["bilstm_model"],
                                  models_bundle["bilstm_tokenizer"], models_bundle["bilstm_max_len"]),
    }

    spam_votes = sum(1 for r in model_results.values() if r["label"] == "spam")
    final_label = "spam" if spam_votes >= 2 else "ham"
    overall_confidence = float(np.mean([r["confidence"] for r in model_results.values() if r["label"] == final_label] or
                                        [r["confidence"] for r in model_results.values()]))

    indicators = detect_spam_indicators(message_text)
    risk_score = calculate_risk_score(model_results, indicators)
    risk_level = interpret_risk(risk_score)
    status = filter_message(final_label, overall_confidence, risk_score)
    recommendation = generate_recommendation(status, indicators)

    return {
        "message": message_text,
        "clean_message": clean_text,
        "model_results": model_results,
        "final_prediction": final_label.upper(),
        "overall_confidence": round(overall_confidence * 100, 2),
        "risk_score": risk_score,
        "risk_level": risk_level,
        "status": status,
        "indicators": indicators,
        "recommendation": recommendation,
    }

print("✅ Spam detection engine, indicator detector, risk scorer, and filter defined.")


In [ ]:
# Quick end-to-end test of the full pipeline
models_bundle = {
    "distilbert_model": distilbert_model, "distilbert_tokenizer": distilbert_tokenizer,
    "bert_model": bert_model, "bert_tokenizer": bert_tokenizer,
    "bilstm_model": bilstm_model, "bilstm_tokenizer": keras_tokenizer, "bilstm_max_len": LSTM_MAX_LEN,
    "device": DEVICE,
}

test_message = "Congratulations! You have won a free prize. Click the link now to claim your reward!"
result = analyze_message(test_message, models_bundle)

print(f"Message:\n{test_message}\n")
for model_name, r in result["model_results"].items():
    print(f"{model_name}: {r['label'].upper()} — {r['confidence']*100:.1f}%")
print(f"\nFinal Prediction: {result['final_prediction']}")
print(f"Overall Confidence: {result['overall_confidence']}%")
print(f"Risk Score: {result['risk_score']}% ({result['risk_level']})")
print(f"Message Status: {result['status']}")
print(f"Indicators: {result['indicators']}")
print(f"Recommendation: {result['recommendation']}")


## Phase 13 — SMS Security Insights & Keyword Analysis

In [ ]:
security_insights = {
    "total_messages": int(total),
    "ham_messages": int(ham_count),
    "spam_messages": int(spam_count),
    "spam_percentage": round(spam_count / total * 100, 2),
    "ham_percentage": round(ham_count / total * 100, 2),
    "avg_message_length": round(df["message_length"].mean(), 2),
    "avg_spam_length": round(df[df["label"] == "spam"]["message_length"].mean(), 2),
    "avg_ham_length": round(df[df["label"] == "ham"]["message_length"].mean(), 2),
}

print("SECURITY INSIGHTS")
for k, v in security_insights.items():
    print(f"  {k}: {v}")

pd.DataFrame([security_insights]).to_csv(PROJECT_ROOT / "results" / "security_insights.csv", index=False)

spam_keywords_df = pd.DataFrame(top_spam_words, columns=["word", "count"])
ham_keywords_df = pd.DataFrame(top_ham_words, columns=["word", "count"])
spam_keywords_df.to_csv(PROJECT_ROOT / "results" / "spam_keywords.csv", index=False)
ham_keywords_df.to_csv(PROJECT_ROOT / "results" / "ham_keywords.csv", index=False)

print("\n✅ Security insights and keyword analysis saved to results/")


## Phase 14 — `requirements.txt` Generation

In [ ]:
requirements_content = """pandas
numpy
scikit-learn
transformers
accelerate
sentence-transformers
torch
tensorflow
streamlit
pyngrok
PyPDF2
nltk
matplotlib
seaborn
wordcloud
kaggle
"""

with open(PROJECT_ROOT / "requirements.txt", "w") as f:
    f.write(requirements_content)

print("✅ requirements.txt generated.")
print(requirements_content)


## Phase 15 — `project_report.md` Generation

The report is generated dynamically from the actual results computed above — nothing is hard-coded.

In [ ]:
report_md = f"""# AI Agent for SMS Spam Detection and Intelligent Message Filtering
### Project Report

Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## 1. Abstract
This project implements an AI-powered SMS spam detection and intelligent message filtering
system. It combines three complementary models — DistilBERT, BERT, and a Bidirectional LSTM —
to classify SMS messages as Spam or Ham, calculates a confidence-weighted risk score, detects
suspicious linguistic patterns, and exposes the entire pipeline through a professional Streamlit
security dashboard.

## 2. Introduction
SMS remains a common channel for phishing, financial fraud, and unsolicited promotional content.
Manually screening large volumes of messages is impractical, motivating an automated, model-driven
approach that flags risky messages and explains *why* they were flagged.

## 3. Problem Statement
Given the volume and evolving nature of spam and phishing SMS content, this project addresses the
need for an automated, explainable system capable of accurately separating spam from legitimate
messages while surfacing the specific signals that drove each decision.

## 4. Objectives
- Automatic Spam/Ham classification of SMS messages
- Comparison of DistilBERT, BERT, and BiLSTM architectures
- Confidence-weighted spam risk scoring (0-100)
- Intelligent message filtering (SAFE / SUSPICIOUS / SPAM)
- Rule-based suspicious indicator detection
- Bulk SMS analysis via CSV upload

## 5. Dataset
- Source: UCI / Kaggle SMS Spam Collection Dataset (`uciml/sms-spam-collection-dataset`)
- Total messages after cleaning: {records_after}
- Spam messages: {(df['label']=='spam').sum()}
- Ham messages: {(df['label']=='ham').sum()}
- Spam percentage: {security_insights['spam_percentage']}%
- Ham percentage: {security_insights['ham_percentage']}%

## 6. Data Preprocessing
Messages were lowercased; URLs, email addresses, HTML tags, and special characters were stripped;
whitespace was normalized; missing and duplicate records were removed ({duplicates_removed}
duplicates removed, {missing_handled} missing/invalid records handled); labels were mapped to
Ham=0 / Spam=1.

## 7. Model Architecture

### DistilBERT
`distilbert-base-uncased` fine-tuned as a binary sequence classifier.

### BERT
`bert-base-uncased` fine-tuned as a binary sequence classifier.

### BiLSTM
```
Input -> Tokenizer -> Embedding -> Bidirectional LSTM -> Dropout -> Dense -> Sigmoid -> Spam/Ham
```

## 8. Model Training
- Max sequence length: {MAX_LENGTH} (BiLSTM: {LSTM_MAX_LEN})
- Batch size: {BATCH_SIZE}
- Transformer epochs: {EPOCHS}
- Device used: {DEVICE}

## 9. Evaluation Metrics
Accuracy, Precision, Recall, F1-score, and confusion matrices were computed for every model on a
held-out stratified test set.

## 10. Results

| Model | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| DistilBERT | {db_accuracy:.4f} | {db_precision:.4f} | {db_recall:.4f} | {db_f1:.4f} |
| BERT | {bert_accuracy:.4f} | {bert_precision:.4f} | {bert_recall:.4f} | {bert_f1:.4f} |
| BiLSTM | {bilstm_accuracy:.4f} | {bilstm_precision:.4f} | {bilstm_recall:.4f} | {bilstm_f1:.4f} |

**Best Model (by F1-score): {BEST_MODEL_NAME} ({BEST_MODEL_F1*100:.2f}%)**

## 11. Spam Analysis
Top spam keywords: {', '.join([w for w, _ in top_spam_words[:10]])}
Average spam message length: {security_insights['avg_spam_length']} characters
Average ham message length: {security_insights['avg_ham_length']} characters

## 12. Intelligent Filtering
Messages are categorized as SAFE, SUSPICIOUS, or SPAM based on the ensemble prediction, its
confidence, and the computed risk score, with configurable confidence thresholds.

## 13. Streamlit Application
`app.py` implements a multi-page dashboard (Home, Analyze SMS, Bulk SMS Analysis, Spam Dashboard,
Model Comparison, Dataset Explorer, About) that loads the saved models directly from
`SMS_Spam_AI_Project/models/` and runs independently of this notebook.

## 14. Conclusion
The ensemble of DistilBERT, BERT, and BiLSTM, combined with rule-based indicator detection, provides
an accurate and explainable spam-filtering system suitable for real-world SMS security triage.

## 15. Future Scope
- Multilingual SMS spam detection
- Real-time SMS monitoring
- Phishing URL analysis and reputation checks
- Explainable AI (attention/SHAP visualizations)
- Transformer ensemble/stacking models
- Voice-message spam detection
- WhatsApp/email spam detection
- Real-time mobile integration
- Continual learning from newly reported spam patterns
"""

with open(PROJECT_ROOT / "project_report.md", "w") as f:
    f.write(report_md)

print(f"✅ project_report.md generated ({len(report_md)} characters).")


## Phase 16 — Generate the Streamlit Application (`app.py`)

This writes a complete, standalone Streamlit dashboard to `SMS_Spam_AI_Project/app.py`. It loads the saved models from disk, so it runs independently of this notebook.

In [ ]:
app_code = r'''import os
import re
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(
    page_title="AI Message Security Dashboard",
    page_icon="\U0001F6E1\uFE0F",
    layout="wide",
    initial_sidebar_state="expanded",
)

PROJECT_ROOT = Path(__file__).resolve().parent
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
CLEANED_DATA_PATH = PROJECT_ROOT / "sms_cleaned.csv"
COMPARISON_PATH = RESULTS_DIR / "model_comparison.csv"

MAX_LENGTH = 128
LSTM_MAX_LEN = 100

# ----------------------------------------------------------------------
# THEME / CSS
# ----------------------------------------------------------------------
CUSTOM_CSS = """
<style>
:root {
    --primary-blue: #1B4F91;
    --accent-blue: #2E86AB;
    --danger-red: #E63946;
    --safe-green: #2A9D8F;
    --warn-amber: #F4A261;
}
.main { background-color: #F7F9FC; }
.kpi-card {
    background: white; border-radius: 12px; padding: 18px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06); border-left: 5px solid var(--accent-blue);
    text-align: center;
}
.kpi-value { font-size: 28px; font-weight: 700; color: var(--primary-blue); }
.kpi-label { font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; }
.model-card {
    background: white; border-radius: 12px; padding: 16px 20px; margin-bottom: 10px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06); border-top: 4px solid var(--accent-blue);
}
.status-spam { background:#FDECEC; color:var(--danger-red); padding:14px; border-radius:10px; font-weight:700; text-align:center; font-size:22px;}
.status-safe { background:#E8F6F3; color:var(--safe-green); padding:14px; border-radius:10px; font-weight:700; text-align:center; font-size:22px;}
.status-suspicious { background:#FEF3E4; color:var(--warn-amber); padding:14px; border-radius:10px; font-weight:700; text-align:center; font-size:22px;}
h1, h2, h3 { color: var(--primary-blue); }
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

# ----------------------------------------------------------------------
# TEXT CLEANING (mirrors the notebook pipeline)
# ----------------------------------------------------------------------
def clean_message(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

SPAM_KEYWORDS = {
    "prize": ["win", "won", "winner", "prize", "lottery", "reward", "jackpot", "claim"],
    "free": ["free", "gift", "bonus", "voucher"],
    "urgent": ["urgent", "immediately", "now", "asap", "act now", "hurry", "limited time", "expire"],
    "financial": ["bank", "account", "credit", "loan", "cash", "money", "payment", "refund", "debit"],
    "otp": ["otp", "verification code", "password", "pin", "verify your account"],
    "promotional": ["offer", "discount", "deal", "sale", "subscribe", "buy now", "click here"],
}

def detect_spam_indicators(message):
    text = str(message)
    lower = text.lower()
    indicators = []
    labels = {
        "prize": "Prize/lottery-related keyword detected",
        "free": "Free-offer keyword detected",
        "urgent": "Urgency detected",
        "financial": "Financial/bank keyword detected",
        "otp": "OTP or account-verification request detected",
        "promotional": "Promotional language detected",
    }
    for category, words in SPAM_KEYWORDS.items():
        if any(w in lower for w in words):
            indicators.append(labels[category])
    if re.search(r"https?://|www\.", lower):
        indicators.append("Suspicious URL detected")
    if re.search(r"\b\d{10,}\b", text) or re.search(r"\b0\d{9,}\b", text):
        indicators.append("Phone number detected")
    if re.search(r"[\u00a3$\u20ac\u20b9]\s?\d", text):
        indicators.append("Currency amount detected")
    letters = [c for c in text if c.isalpha()]
    if letters and sum(1 for c in letters if c.isupper()) / len(letters) > 0.4 and len(letters) > 8:
        indicators.append("Excessive capital letters detected")
    if text.count("!") >= 2 or text.count("?") >= 2:
        indicators.append("Excessive punctuation detected")
    return indicators

def calculate_risk_score(model_results, indicators):
    spam_confidences = [r["confidence"] for r in model_results.values() if r["label"] == "spam"]
    ham_confidences = [r["confidence"] for r in model_results.values() if r["label"] == "ham"]
    n_models = len(model_results)
    spam_votes = len(spam_confidences)
    if spam_votes == 0:
        base_score = (1 - np.mean(ham_confidences)) * 40
    else:
        base_score = (spam_votes / n_models) * 60 + np.mean(spam_confidences) * 30
    indicator_bonus = min(len(indicators) * 4, 15)
    return round(float(min(base_score + indicator_bonus, 100)), 2)

def interpret_risk(score):
    if score <= 30:
        return "LOW RISK"
    elif score <= 60:
        return "MEDIUM RISK"
    elif score <= 80:
        return "HIGH RISK"
    return "VERY HIGH RISK"

def filter_message(final_label, overall_confidence, safe_threshold=0.75, spam_threshold=0.75):
    if final_label == "ham" and overall_confidence >= safe_threshold:
        return "SAFE"
    if final_label == "spam" and overall_confidence >= spam_threshold:
        return "SPAM"
    return "SUSPICIOUS"

def generate_recommendation(status):
    if status == "SPAM":
        return ("Avoid clicking any links in this message. Do not provide OTP, passwords, "
                "bank details, or personal information. Consider blocking the sender.")
    elif status == "SUSPICIOUS":
        return ("This message shows some suspicious characteristics. Verify the sender through "
                "an official channel before responding or clicking any links.")
    return "This message appears safe based on the trained models."

# ----------------------------------------------------------------------
# MODEL LOADING (cached)
# ----------------------------------------------------------------------
@st.cache_resource(show_spinner="Loading AI models...")
def load_models():
    bundle = {"error": None}
    try:
        import torch
        from transformers import (
            DistilBertTokenizerFast, DistilBertForSequenceClassification,
            BertTokenizerFast, BertForSequenceClassification,
        )
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        db_dir = MODELS_DIR / "distilbert"
        bert_dir = MODELS_DIR / "bert"
        bilstm_dir = MODELS_DIR / "bilstm"

        if db_dir.exists():
            bundle["distilbert_tokenizer"] = DistilBertTokenizerFast.from_pretrained(db_dir)
            bundle["distilbert_model"] = DistilBertForSequenceClassification.from_pretrained(db_dir).to(device)
            bundle["distilbert_model"].eval()
        if bert_dir.exists():
            bundle["bert_tokenizer"] = BertTokenizerFast.from_pretrained(bert_dir)
            bundle["bert_model"] = BertForSequenceClassification.from_pretrained(bert_dir).to(device)
            bundle["bert_model"].eval()
        if bilstm_dir.exists():
            import tensorflow as tf
            bundle["bilstm_model"] = tf.keras.models.load_model(bilstm_dir / "bilstm_model.h5")
            with open(bilstm_dir / "tokenizer.pkl", "rb") as f:
                bundle["bilstm_tokenizer"] = pickle.load(f)
            with open(bilstm_dir / "config.json") as f:
                bundle["bilstm_max_len"] = json.load(f).get("max_len", LSTM_MAX_LEN)

        bundle["device"] = device
    except Exception as e:
        bundle["error"] = str(e)
    return bundle

def predict_transformer(message, model, tokenizer, device):
    import torch
    inputs = tokenizer(message, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_class = int(np.argmax(probs))
    return {"label": "spam" if pred_class == 1 else "ham", "confidence": float(probs[pred_class])}

def predict_bilstm(message, model, tokenizer, max_len):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    seq = tokenizer.texts_to_sequences([message])
    padded = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0).flatten()[0])
    label = "spam" if prob >= 0.5 else "ham"
    confidence = prob if label == "spam" else 1 - prob
    return {"label": label, "confidence": confidence}

def analyze_message(message_text, bundle):
    if not message_text or not str(message_text).strip():
        return {"error": "Please enter a valid, non-empty SMS message."}

    clean_text = clean_message(message_text)
    model_results = {}

    if bundle.get("distilbert_model"):
        model_results["DistilBERT"] = predict_transformer(
            clean_text, bundle["distilbert_model"], bundle["distilbert_tokenizer"], bundle["device"])
    if bundle.get("bert_model"):
        model_results["BERT"] = predict_transformer(
            clean_text, bundle["bert_model"], bundle["bert_tokenizer"], bundle["device"])
    if bundle.get("bilstm_model"):
        model_results["BiLSTM"] = predict_bilstm(
            clean_text, bundle["bilstm_model"], bundle["bilstm_tokenizer"], bundle["bilstm_max_len"])

    if not model_results:
        return {"error": "No trained models were found. Please run the training notebook first."}

    spam_votes = sum(1 for r in model_results.values() if r["label"] == "spam")
    final_label = "spam" if spam_votes >= (len(model_results) / 2) else "ham"
    matching = [r["confidence"] for r in model_results.values() if r["label"] == final_label]
    overall_confidence = float(np.mean(matching)) if matching else float(np.mean([r["confidence"] for r in model_results.values()]))

    indicators = detect_spam_indicators(message_text)
    risk_score = calculate_risk_score(model_results, indicators)
    risk_level = interpret_risk(risk_score)
    status = filter_message(final_label, overall_confidence)
    recommendation = generate_recommendation(status)

    return {
        "model_results": model_results,
        "final_prediction": final_label.upper(),
        "overall_confidence": round(overall_confidence * 100, 2),
        "risk_score": risk_score,
        "risk_level": risk_level,
        "status": status,
        "indicators": indicators,
        "recommendation": recommendation,
    }

# ----------------------------------------------------------------------
# DATA LOADING
# ----------------------------------------------------------------------
@st.cache_data
def load_cleaned_data():
    if CLEANED_DATA_PATH.exists():
        return pd.read_csv(CLEANED_DATA_PATH)
    return pd.DataFrame(columns=["label", "message", "clean_message"])

@st.cache_data
def load_comparison():
    if COMPARISON_PATH.exists():
        return pd.read_csv(COMPARISON_PATH)
    return pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"])

data_df = load_cleaned_data()
comparison_df = load_comparison()
best_model_name = comparison_df.loc[comparison_df["F1-Score"].idxmax(), "Model"] if not comparison_df.empty else "N/A"

# ----------------------------------------------------------------------
# SIDEBAR NAVIGATION
# ----------------------------------------------------------------------
st.sidebar.markdown("## \U0001F6E1\uFE0F AI Message Security")
page = st.sidebar.radio(
    "Navigation",
    ["Home", "Analyze SMS", "Bulk SMS Analysis", "Spam Dashboard", "Model Comparison", "Dataset Explorer", "About Project"],
)
st.sidebar.markdown("---")
if st.sidebar.button("\U0001F50D Analyze New Message"):
    page = "Analyze SMS"

# ----------------------------------------------------------------------
# HOME PAGE
# ----------------------------------------------------------------------
if page == "Home":
    st.title("AI SMS Spam Detection & Intelligent Message Filtering")
    st.subheader("Protect Your Messages with AI-Powered Spam Detection")

    total_msgs = len(data_df)
    spam_msgs = (data_df["label"] == "spam").sum() if "label" in data_df else 0
    spam_pct = round(spam_msgs / total_msgs * 100, 1) if total_msgs else 0

    c1, c2, c3, c4, c5 = st.columns(5)
    for col, label, value in zip(
        [c1, c2, c3, c4, c5],
        ["Total Messages", "Spam Messages", "Safe Messages", "Spam %", "Best Model"],
        [total_msgs, spam_msgs, total_msgs - spam_msgs, f"{spam_pct}%", best_model_name],
    ):
        col.markdown(f'<div class="kpi-card"><div class="kpi-value">{value}</div><div class="kpi-label">{label}</div></div>', unsafe_allow_html=True)

    st.markdown("### Project Overview")
    st.write(
        "This platform analyzes SMS messages using three AI models — **DistilBERT**, **BERT**, and "
        "**BiLSTM** — to detect spam, calculate a risk score, identify suspicious patterns, and "
        "provide intelligent filtering with clear security recommendations."
    )
    st.markdown("### Models Used")
    st.write("- DistilBERT (`distilbert-base-uncased`)\n- BERT (`bert-base-uncased`)\n- Bidirectional LSTM (TensorFlow/Keras)")

    if st.button("\U0001F680 Analyze SMS Message", type="primary"):
        st.info("Use the **Analyze SMS** page from the sidebar to analyze a message.")

# ----------------------------------------------------------------------
# ANALYZE SMS PAGE
# ----------------------------------------------------------------------
elif page == "Analyze SMS":
    st.title("\U0001F50D SMS Analysis")
    bundle = load_models()
    if bundle.get("error"):
        st.error(f"Model loading error: {bundle['error']}")

    message = st.text_area(
        "Enter SMS Message",
        placeholder="Congratulations! You have won a free prize. Click the link now.",
        height=120,
    )

    if st.button("Analyze Message", type="primary"):
        if not message.strip():
            st.warning("Please enter a message to analyze.")
        else:
            result = analyze_message(message, bundle)
            if "error" in result:
                st.error(result["error"])
            else:
                st.markdown("### Model Predictions")
                cols = st.columns(len(result["model_results"]))
                for col, (name, r) in zip(cols, result["model_results"].items()):
                    col.markdown(
                        f'<div class="model-card"><b>{name}</b><br>'
                        f'Prediction: <b>{r["label"].upper()}</b><br>'
                        f'Confidence: <b>{r["confidence"]*100:.1f}%</b></div>',
                        unsafe_allow_html=True,
                    )

                st.markdown("### Final Prediction")
                st.write(f"**Final Prediction:** {result['final_prediction']}  |  **Overall Confidence:** {result['overall_confidence']}%")

                st.markdown("### Spam Risk Score")
                st.progress(int(result["risk_score"]))
                st.write(f"**{result['risk_score']}% — {result['risk_level']}**")

                st.markdown("### Intelligent Message Filtering")
                css_class = {"SPAM": "status-spam", "SAFE": "status-safe", "SUSPICIOUS": "status-suspicious"}[result["status"]]
                st.markdown(f'<div class="{css_class}">Message Status: {result["status"]}</div>', unsafe_allow_html=True)

                if result["indicators"]:
                    st.markdown("**Detected Indicators:**")
                    for ind in result["indicators"]:
                        st.write(f"- {ind}")
                else:
                    st.write("No suspicious indicators detected.")

                st.markdown("### Security Recommendation")
                st.info(result["recommendation"])

# ----------------------------------------------------------------------
# BULK SMS ANALYSIS PAGE
# ----------------------------------------------------------------------
elif page == "Bulk SMS Analysis":
    st.title("\U0001F4CB Bulk SMS Analysis")
    bundle = load_models()
    st.write("Upload a CSV file with a `message` column to analyze multiple messages at once.")

    uploaded_file = st.file_uploader("Upload CSV", type=["csv"])
    if uploaded_file is not None:
        try:
            bulk_df = pd.read_csv(uploaded_file)
        except Exception as e:
            st.error(f"Could not read the uploaded CSV: {e}")
            bulk_df = None

        if bulk_df is not None:
            msg_col = None
            for c in bulk_df.columns:
                if c.strip().lower() in ("message", "sms", "text"):
                    msg_col = c
                    break
            if msg_col is None:
                st.error("The uploaded CSV must contain a 'message' column.")
            else:
                with st.spinner("Analyzing messages..."):
                    rows = []
                    for msg in bulk_df[msg_col].astype(str):
                        res = analyze_message(msg, bundle)
                        if "error" in res:
                            rows.append({"Message": msg, "Prediction": "N/A", "Confidence": "N/A", "Risk": "N/A"})
                        else:
                            rows.append({
                                "Message": msg,
                                "Prediction": res["final_prediction"],
                                "Confidence": f"{res['overall_confidence']}%",
                                "Risk": f"{res['risk_score']}%",
                            })
                    results_table = pd.DataFrame(rows)

                st.dataframe(results_table, use_container_width=True)
                csv_bytes = results_table.to_csv(index=False).encode("utf-8")
                st.download_button("\U0001F4E5 Download Filtered Results", data=csv_bytes, file_name="filtered_results.csv", mime="text/csv")

# ----------------------------------------------------------------------
# SPAM DASHBOARD PAGE
# ----------------------------------------------------------------------
elif page == "Spam Dashboard":
    st.title("\U0001F4CA Spam Dashboard")

    if data_df.empty:
        st.warning("No dataset found. Please run the training notebook first.")
    else:
        total_msgs = len(data_df)
        spam_msgs = (data_df["label"] == "spam").sum()
        ham_msgs = total_msgs - spam_msgs
        spam_pct = round(spam_msgs / total_msgs * 100, 1)

        c1, c2, c3, c4, c5 = st.columns(5)
        for col, label, value in zip(
            [c1, c2, c3, c4, c5],
            ["Total Messages", "Spam Messages", "Safe Messages", "Spam %", "Avg Risk Score"],
            [total_msgs, spam_msgs, ham_msgs, f"{spam_pct}%", "See Analyze SMS"],
        ):
            col.markdown(f'<div class="kpi-card"><div class="kpi-value">{value}</div><div class="kpi-label">{label}</div></div>', unsafe_allow_html=True)

        colA, colB = st.columns(2)
        with colA:
            fig, ax = plt.subplots()
            sns.countplot(x=data_df["label"], ax=ax, palette=["#2E86AB", "#E63946"])
            ax.set_title("Spam vs Ham Distribution")
            st.pyplot(fig)
        with colB:
            fig2, ax2 = plt.subplots()
            lengths = data_df["message"].astype(str).apply(len)
            sns.histplot(lengths, bins=30, ax=ax2, color="#2E86AB")
            ax2.set_title("Message Length Distribution")
            st.pyplot(fig2)

# ----------------------------------------------------------------------
# MODEL COMPARISON PAGE
# ----------------------------------------------------------------------
elif page == "Model Comparison":
    st.title("\U0001F9E0 Model Comparison")
    st.subheader("DistilBERT vs BERT vs BiLSTM")

    if comparison_df.empty:
        st.warning("No model comparison results found. Please run the training notebook first.")
    else:
        st.dataframe(comparison_df, use_container_width=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        comparison_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-Score"]].plot(kind="bar", ax=ax)
        plt.xticks(rotation=0)
        st.pyplot(fig)
        st.success(f"\U0001F3C6 Best Performing Model: **{best_model_name}**")

# ----------------------------------------------------------------------
# DATASET EXPLORER PAGE
# ----------------------------------------------------------------------
elif page == "Dataset Explorer":
    st.title("\U0001F4C2 Dataset Explorer")

    if data_df.empty:
        st.warning("No dataset found. Please run the training notebook first.")
    else:
        search = st.text_input("Search messages")
        label_filter = st.selectbox("Filter by label", ["All", "spam", "ham"])

        filtered = data_df.copy()
        if search:
            filtered = filtered[filtered["message"].astype(str).str.contains(search, case=False, na=False)]
        if label_filter != "All":
            filtered = filtered[filtered["label"] == label_filter]

        filtered = filtered.copy()
        filtered["message_length"] = filtered["message"].astype(str).apply(len)
        st.dataframe(filtered[["label", "message", "message_length"]], use_container_width=True)

# ----------------------------------------------------------------------
# ABOUT PROJECT PAGE
# ----------------------------------------------------------------------
elif page == "About Project":
    st.title("\u2139\uFE0F About This Project")
    st.markdown("### Project Title")
    st.write("AI Agent for SMS Spam Detection and Intelligent Message Filtering")

    st.markdown("### Technologies")
    st.write("Python, NLP, BERT, DistilBERT, BiLSTM, Transformers, TensorFlow/PyTorch, Streamlit, Google Colab, ngrok")

    st.markdown("### Dataset")
    st.write("UCI SMS Spam Collection / Kaggle SMS Spam Collection Dataset (`uciml/sms-spam-collection-dataset`)")

    report_path = PROJECT_ROOT / "project_report.md"
    if report_path.exists():
        with open(report_path, "rb") as f:
            st.download_button("\U0001F4C4 Download AI Spam Detection & Security Report", data=f, file_name="project_report.md")
'''

app_path = PROJECT_ROOT / "app.py"
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"✅ app.py generated at: {app_path}")
print(f"File size: {app_path.stat().st_size} bytes")


## Phase 17 — Launch Streamlit via ngrok

Enter your ngrok authtoken when prompted (from https://dashboard.ngrok.com/get-started/your-authtoken). It is never hard-coded or written to a shared file.

In [ ]:
from pyngrok import ngrok, conf
import getpass

try:
    ngrok_token = getpass.getpass("Enter your ngrok authtoken (input hidden): ").strip()
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print("✅ ngrok authtoken configured.")
    else:
        print("⚠️ No token entered — ngrok tunnels require a free authtoken from https://dashboard.ngrok.com/")
except Exception as e:
    print(f"❌ ngrok configuration failed: {e}")


In [ ]:
import subprocess, time

# Kill any previous Streamlit/ngrok processes
os.system("pkill -f streamlit || true")
try:
    ngrok.kill()
except Exception:
    pass

# Launch Streamlit in the background
streamlit_process = subprocess.Popen([
    "streamlit", "run", str(PROJECT_ROOT / "app.py"),
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
])

time.sleep(8)

try:
    public_url = ngrok.connect(8501, "http")
    print("=" * 55)
    print("✅ Streamlit application is running.")
    print(f"Public URL: {public_url}")
    print("=" * 55)
except Exception as e:
    print(f"❌ ngrok tunnel failed to start: {e}")
    print("Check that your ngrok authtoken was entered correctly above.")


## Phase 18 — Package & Download the Complete Project

In [ ]:
# Create a ZIP archive of the entire project
zip_path = "SMS_Spam_AI_Project.zip"

shutil.make_archive("SMS_Spam_AI_Project", "zip", PROJECT_ROOT)
print(f"✅ Project packaged: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")


In [ ]:
# Exact commands to download individual deliverables from Google Colab
from google.colab import files as colab_files

print("Run any of the following in a new cell to download that specific deliverable:\n")
print('files.download("SMS_Spam_AI_Project.zip")           # Complete project ZIP')
print('files.download("SMS_Spam_AI_Project/app.py")         # Streamlit app')
print('files.download("SMS_Spam_AI_Project/sms_cleaned.csv") # Cleaned dataset')
print('files.download("SMS_Spam_AI_Project/project_report.md") # Project report')
print('files.download("SMS_Spam_AI_Project/requirements.txt")  # Requirements file')
print("\nFor entire folders (models/, results/), zip them individually, e.g.:")
print('!zip -r models.zip SMS_Spam_AI_Project/models')
print('files.download("models.zip")')

# Uncomment the next line to trigger the ZIP download immediately:
# colab_files.download("SMS_Spam_AI_Project.zip")


---
## ✅ Project Complete

You now have a fully trained, compared, and deployed **AI SMS Spam Detection & Intelligent Message Filtering** system:

- Trained **DistilBERT**, **BERT**, and **BiLSTM** models saved under `SMS_Spam_AI_Project/models/`
- A dynamically generated **model comparison** and **project report**
- A professional **Streamlit Message Security Dashboard** (`app.py`) exposed publicly via **ngrok**
- A complete, downloadable **project ZIP** (`SMS_Spam_AI_Project.zip`)

To restart the dashboard later without retraining, just re-run the folder-structure, model-loading (Phase 17 cells), and ngrok-launch cells — `app.py` loads the already-saved models from disk.
